In [52]:
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt

In [53]:
# fator de segurança
FC = 1.0
# quantidade de bancos em paralelo
nr_bancos = 5
# tensão fase-fase
V_ff = 23.1e3
# tensão fase neutro
V_fn = V_ff / np.sqrt(3)
# frequência fundamental
f_fund = 60
w_fund = 2*np.pi*f_fund
# potência trifásica do banco
Q_3f = 30e6 * np.ones(nr_bancos, dtype=float)
Q_1f = Q_3f / 3
# corrente de fase dos estágios do banco em Y
I_fn =  Q_1f / V_fn
print(f"Corrente de fase dos estágios do banco em Y = {I_fn} A")

Corrente de fase dos estágios do banco em Y = [749.8055444 749.8055444 749.8055444 749.8055444 749.8055444] A


In [54]:
# reatância e capacitância dos estágios
X = V_fn / I_fn
C = 1 / ( w_fund * X )
print(f"Corrente de fase dos estágios do banco em Y = {1e6*C} uF")

Corrente de fase dos estágios do banco em Y = [149.13039775 149.13039775 149.13039775 149.13039775 149.13039775] uF


In [55]:
# Henries por metro da barra
L_unit_barra = 0.0329e-6 * np.ones(nr_bancos, dtype=float)
# Henries por metro do cabo
L_unit_cabo  = 0.396e-6  * np.zeros(nr_bancos, dtype=float)

# comprimento da barra
comp_barra = 30.0 * np.zeros(nr_bancos, dtype=float)
# comprimento do cabo
comp_cabo  = 20.0 * np.ones(nr_bancos, dtype=float)

# indutância série da barra e do cabo
L_barra_mais_cabo = comp_barra * L_unit_barra + comp_cabo * L_unit_cabo

In [56]:
# alguns capacitores tem uma pequena indutância em série de fábrica
L_capacitor = 5e-6 * np.ones(nr_bancos, dtype=float)
# indutor a ser colocado em série com o estágio capacitivo
L_reator = 200e-6 *np.ones(nr_bancos, dtype=float)

In [57]:
# indutância total entre dois estágios de capacitor
L = L_barra_mais_cabo + L_capacitor + L_reator
print(f"Indutância total entre dois estágios de capacitor = {1e6*L} uH")

Indutância total entre dois estágios de capacitor = [205. 205. 205. 205. 205.] uH


In [58]:
from IPython.core.display import Latex
# imagiando todos os "n-1" capacitores ligados e entrando o capacitor "0"
# capacitância combinada dos "n-1" capacitores ligados
C_paralelos = np.sum(C[1:])
# associando estes "n-1" em série com o capacitor a ser acionado
den_C = 1/C[0] + 1/C_paralelos
C_eq = 1/den_C

# indutância combinada dos "n-1" capacitores ligados
L_paralelos =  1 / np.sum( 1 / L[1:] )
# associando estes "n-1" em série com a indutância do capacitor a ser acionado
L_eq = L[0] + L_paralelos
# resistência imaginada do sistema
R_eq = 0.1

print(f"Capacitância dos 'n-1' capacitores em paralelo = {1e6*C_paralelos} uF")
print(f"Capacitância do capacitor a ser acionado       = {1e6*C[0]} uF")
print(f"Capacitância C[1:] série com C[0]              = {1e6*C_eq} uF")
print("")
print(f"Indutância combinada dos 'n-1' capacitores ligados = {1e6*L_paralelos} uH")
print(f"Indutância do capacitor a ser acionado             = {1e6*L[0]} uH")
print(f"Indutância L[1:] série com L[0]                    = {1e6*L_eq} uH")

Capacitância dos 'n-1' capacitores em paralelo = 596.5215910192662 uF
Capacitância do capacitor a ser acionado       = 149.13039775481656 uF
Capacitância C[1:] série com C[0]              = 119.30431820385324 uF

Indutância combinada dos 'n-1' capacitores ligados = 51.250000000000014 uH
Indutância do capacitor a ser acionado             = 205.00000000000003 uH
Indutância L[1:] série com L[0]                    = 256.25 uH


$$I\left( s \right) = \frac{{\left( {\frac{{\sqrt 2 V}}{s}} \right)}}{{\left( {R + sL + \frac{1}{{sC}}} \right)}} = \frac{{\sqrt 2 V}}{{s\left( {Ls + R + \frac{1}{{Cs}}} \right)}}$$
$$ $$
$$i\left( t \right) = \frac{{2\sqrt 2 \sqrt C V{e^{ - \frac{{Rt}}{{2L}}}}\sin \left( {\frac{{t\sqrt { - C{R^2} + 4L} }}{{2\sqrt C L}}} \right)}}{{\sqrt { - C{R^2} + 4L} }}\theta \left( t \right)$$

In [59]:
raiz = -(R_eq/L_eq)**2 +  4/(C_eq*L_eq)
omega = np.sqrt(raiz) / 2
num_i = V_fn * np.sqrt(2)
den_i = L_eq * omega
i_pico_inical =  FC * num_i / den_i
sigma = -R_eq/(2*L_eq)

In [60]:
t = np.linspace(0, 2/60, 2*(2**12) )
i_curto = i_pico_inical * np.exp(sigma*t) * np.sin(omega*t)

In [61]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t*1e3,
    y=i_curto/1e3,
    name="Instantânea"
))

fig.add_trace(go.Scatter(
    x=t*1e3,
    y=i_pico_inical * np.exp(sigma*t)/1e3,
    name="Envelope",
    line = dict(shape = 'linear', color = 'rgb(0, 0, 0)', width = 1, dash = 'dot'),
    connectgaps = True)
)

fig.add_trace(go.Scatter(
    x=t*1e3,
    y=-i_pico_inical * np.exp(sigma*t)/1e3,
    name="Envelope",
    line = dict(shape = 'linear', color = 'rgb(0, 0, 0)', width = 1, dash = 'dot'),
    connectgaps = True)
)

fig.update_layout(legend_title_text='Corrente:', title_text="Inrush Banco de Capacitores",
                  xaxis_title=r"Tempo [ms]", yaxis_title="Corrente [kA]")
fig.show()

In [62]:
print("Corrente de pico considerada = ", np.round(i_pico_inical/1e3, 2), "kA")
print("Corrente de pico real = ", np.round(np.max(i_curto)/1e3, 2), "kA")
print("Ireal/Iconsiderado =", np.round(100*np.max(i_curto)/i_pico_inical, 2), "%")
print("Frequência de Oscilação = ", np.round(omega/(2*np.pi), 2), "Hz")
print("Harmônico de Oscilação = ", np.round(omega/w_fund, 2))


Corrente de pico considerada =  12.88 kA
Corrente de pico real =  12.21 kA
Ireal/Iconsiderado = 94.83 %
Frequência de Oscilação =  909.72 Hz
Harmônico de Oscilação =  15.16
